In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.7 MB/s eta 0:00:00


In [ ]:
!unzip /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_Sheets_TestOnly.zip

Archive:  /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_Sheets_TestOnly.zip
   creating: content/OMR_5Fold_Sheets_TestOnly/
   creating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/
   creating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/
   creating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_57_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_59_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_60_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_22_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_20_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_2_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_6_1.png  
  inflating: content/OMR_5Fold_Sheets_TestOnly/Fold_4/test/exam3/exam3_37_1.png  
  inflating: content/OMR_5Fold_Sheets_Tes

In [ ]:
import scipy.io
import numpy as np
import pandas as pd
import os
import cv2
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from ultralytics import YOLO

# =============================================================================
# --- 1. CẤU HÌNH ĐƯỜNG DẪN 5 FOLDS ---
# =============================================================================
MODEL_METADATA_FILE = '/content/drive/MyDrive/OMR-Datasets/metadata/modelAnswerMetadata.mat'
EXAMS_METADATA_FILE = '/content/drive/MyDrive/OMR-Datasets/metadata/exams.mat'

REPORT_OUTPUT_FILE = '/content/drive/MyDrive/OMR-Datasets/report/omr_5fold_detailed_evaluation_eff_gan.csv'
SUMMARY_OUTPUT_FILE = '/content/drive/MyDrive/OMR-Datasets/report/omr_5fold_final_summary_eff_gan.csv'

# Thư mục gốc chứa 5 Fold
BASE_FOLDS_DIR = '/content/content/OMR_5Fold_Sheets_TestOnly'
WEIGHT_DIR_CLS = '/content/drive/MyDrive/OMR-Datasets/train-cls/scene2/EfficientNetB0'  # <===== Đổi kịch bản tại đây

MODEL_TYPE = 'efficientnet'  # 'yolo' hoặc 'efficientnet'
TARGET_EXAM_ID = None
GT_CODE_MAP = {1: 'confirmed', 2: 'crossedout', 3: 'empty'}
NUM_CLASSES = 3
CLASS_NAMES = ['confirmed', 'crossedout', 'empty']
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

eff_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# =============================================================================
# --- 2. CÁC HÀM XỬ LÝ ---
# =============================================================================
def clean_mat_string(numpy_element):
    try:
        while isinstance(numpy_element, np.ndarray) and numpy_element.size > 0:
            numpy_element = numpy_element[0]
        return str(numpy_element).strip()
    except:
        return ""

def pad_to_square_cv2(img, fill_color=(255, 255, 255)):
    h, w = img.shape[:2]
    max_wh = max(w, h)
    top = (max_wh - h) // 2
    bottom = max_wh - h - top
    left = (max_wh - w) // 2
    right = max_wh - w - left
    return cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=fill_color)

def extract_rois_cv2(image_path, question_rects, question_configs):
    img = cv2.imread(image_path)
    if img is None: return None, None, None

    question_rects = np.array(question_rects)
    question_rects = np.squeeze(question_rects)
    if question_rects.size % 4 == 0: question_rects = question_rects.reshape(-1, 4)

    roi_images, roi_labels, roi_indices = [], [], []
    current_rect_idx = 0

    for q_config in question_configs:
        for opt in q_config['options']:
            if current_rect_idx >= len(question_rects): break

            rect = question_rects[current_rect_idx]
            x, y, w, h = int(rect[0]), int(rect[1]), int(rect[2]), int(rect[3])
            roi = img[y:y+h, x:x+w]

            if roi.size > 0:
                roi_padded = pad_to_square_cv2(roi)
                roi_resized = cv2.resize(roi_padded, (128, 128))
                roi_images.append(roi_resized)
                roi_labels.append(f"{q_config['q_name']}_{opt}")
                roi_indices.append(current_rect_idx)
            current_rect_idx += 1

    return roi_images, roi_labels, roi_indices

def load_all_model_metadata(mat_file_path):
    print(f"Loading Metadata from {mat_file_path}...")
    try:
        data = scipy.io.loadmat(mat_file_path)
        model_answers = data['modelAnswer']
    except Exception as e:
        print(f"Lỗi đọc model: {e}")
        return {}

    all_models = {}
    for i in range(model_answers.shape[1]):
        entry = model_answers[0, i]
        exam_id = int(clean_mat_string(entry[1]).replace('exam', '')) if 'exam' in clean_mat_string(entry[1]) else 0
        page_num = int(entry[2][0,0])
        options_per_question = entry[7][0]
        correct_answers = entry[8][0]

        question_configs = []
        for q_idx, (ans_code, num_opts) in enumerate(zip(correct_answers, options_per_question)):
            q_name = f"Q{q_idx+1}"
            if exam_id == 5:
                num_opts = 4 if q_idx < 10 else 2
            question_configs.append({
                'q_name': q_name,
                'options': [chr(65+k) for k in range(num_opts)]
            })

        if exam_id not in all_models: all_models[exam_id] = {}
        all_models[exam_id][page_num] = {'question_configs': question_configs}
    return all_models

def load_efficientnet_model(model_path):
    print(f"Loading EfficientNet-B0 from {model_path}...")
    model = models.efficientnet_b0(weights=None)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, NUM_CLASSES)
    try:
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    except Exception as e:
        print(f"❌ Lỗi load file .pth: {e}")
        return None
    model.to(DEVICE)
    model.eval()
    return model

def predict_batch_efficientnet(model, roi_imgs_list):
    if not roi_imgs_list: return []
    batch_tensors = []
    for img in roi_imgs_list:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = eff_transforms(img_rgb)
        batch_tensors.append(tensor)

    input_batch = torch.stack(batch_tensors).to(DEVICE)
    with torch.no_grad():
        outputs = model(input_batch)
        _, preds = torch.max(outputs, 1)

    results = [CLASS_NAMES[idx] for idx in preds.cpu().numpy()]
    return results

def evaluate_recognition_performance(student_answers, gt_answer_types, question_configs, roi_labels, roi_indices):
    question_stats = {
        'total_questions': 0,
        'correct_questions': 0,
        'wrong_details': []
    }
    label_to_gt_idx = {label: idx for label, idx in zip(roi_labels, roi_indices)}

    for q_config in question_configs:
        q_name = q_config['q_name']
        options = q_config['options']

        gt_choices = set()
        for opt in options:
            key = f"{q_name}_{opt}"
            if key in label_to_gt_idx:
                gt_idx = label_to_gt_idx[key]
                if gt_idx < len(gt_answer_types):
                    gt_code = gt_answer_types[gt_idx]
                    if gt_code == 1:
                        gt_choices.add(opt)

        pred_choices = set()
        for opt in options:
            key = f"{q_name}_{opt}"
            if student_answers.get(key) == 'confirmed':
                pred_choices.add(opt)

        question_stats['total_questions'] += 1
        if gt_choices == pred_choices:
            question_stats['correct_questions'] += 1
        else:
            gt_str = ",".join(sorted(gt_choices)) if gt_choices else "Empty"
            pred_str = ",".join(sorted(pred_choices)) if pred_choices else "Empty"
            question_stats['wrong_details'].append(f"{q_name}: GT=[{gt_str}] vs AI=[{pred_str}]")

    return question_stats

def save_debug_rois(roi_images, roi_labels, image_name, output_dir='debug_rois'):
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    clean_name = os.path.splitext(image_name)[0]
    sheet_dir = os.path.join(output_dir, clean_name)
    if not os.path.exists(sheet_dir): os.makedirs(sheet_dir)

    for img, label in zip(roi_images, roi_labels):
        filename = f"{label}.jpg"
        save_path = os.path.join(sheet_dir, filename)
        if img is not None and img.size > 0:
            cv2.imwrite(save_path, img)

# =============================================================================
# --- 3. VÒNG LẶP CHÍNH QUA 5 FOLDS ---
# =============================================================================
def main():
    print("🚀 BẮT ĐẦU ĐÁNH GIÁ 5-FOLD CROSS VALIDATION TRÊN TOÀN PHIẾU...")
    all_models_meta = load_all_model_metadata(MODEL_METADATA_FILE)

    print(f"Loading Student Exams from {EXAMS_METADATA_FILE}...")
    try:
        exams_data = scipy.io.loadmat(EXAMS_METADATA_FILE)['records'][0]
    except Exception as e:
        print(f"Lỗi mở file exams.mat: {e}")
        return

    fold_results = []

    all_detailed_reports = []

    for fold in range(1, 6):
        print(f"\n{'='*60}")
        print(f"🔍 ĐANG ĐÁNH GIÁ FOLD {fold} / 5")
        print(f"{'='*60}")

        fold_dir = os.path.join(BASE_FOLDS_DIR, f"Fold_{fold}")
        current_test_dir = os.path.join(fold_dir, "test")
        cur_weight_dir = os.path.join(WEIGHT_DIR_CLS, f"Fold_{fold}")

        # Load Model
        classify_model = None
        if MODEL_TYPE == 'yolo':
            yolo_path = os.path.join(cur_weight_dir, "weights", "best.pt")
            if not os.path.exists(yolo_path):
                print(f"❌ Không tìm thấy YOLO model tại {yolo_path}. Bỏ qua Fold {fold}.")
                continue
            classify_model = YOLO(yolo_path)

        elif MODEL_TYPE == 'efficientnet':
            eff_path = os.path.join(cur_weight_dir, f"efficientnet_b0_fold{fold}_gan.pth")
            classify_model = load_efficientnet_model(eff_path)
            if classify_model is None: continue

        fold_total_q, fold_correct_q = 0, 0
        actual_total_sheets, actual_perfect_sheets = 0, 0

        exam0_sheets = {}

        for sheet in exams_data:
            try:
                if 'examId' in sheet.dtype.names:
                    exam_id = clean_mat_string(sheet['examId'])
                else:
                    exam_id = 0

                try: exam_id = int(float(exam_id))
                except: exam_id = 0

                if TARGET_EXAM_ID is not None and exam_id != TARGET_EXAM_ID: continue

                page_num = int(clean_mat_string(sheet['pageNumber'])) if 'pageNumber' in sheet.dtype.names else 1
                image_name = clean_mat_string(sheet['imageName'])
                gt_answer_types = sheet[2].flatten() if sheet[2].size > 0 else []

                q_rect_raw = sheet['questionRect']
                while isinstance(q_rect_raw, np.ndarray) and q_rect_raw.ndim > 2: q_rect_raw = q_rect_raw[0]
                if q_rect_raw.ndim == 2 and q_rect_raw.shape[0] == 1: q_rect_raw = q_rect_raw[0]
                question_rects = q_rect_raw
            except Exception: continue

            # KIỂM TRA PHIẾU CÓ THUỘC FOLD HIỆN TẠI KHÔNG
            full_image_path = os.path.join(current_test_dir, f"exam{exam_id}", image_name)
            if not os.path.exists(full_image_path):
                full_image_path_root = os.path.join(current_test_dir, image_name)
                if os.path.exists(full_image_path_root):
                    full_image_path = full_image_path_root
                else:
                    continue

            if exam_id not in all_models_meta: continue
            q_configs = all_models_meta[exam_id][page_num]['question_configs']

            roi_imgs, roi_labels, roi_indices = extract_rois_cv2(full_image_path, question_rects, q_configs)
            if not roi_imgs: continue

            student_answers = {}
            if MODEL_TYPE == 'yolo':
                preds = classify_model(roi_imgs, verbose=False)
                for idx, res in enumerate(preds):
                    student_answers[roi_labels[idx]] = res.names[res.probs.top1]

            elif MODEL_TYPE == 'efficientnet':
                preds_list = predict_batch_efficientnet(classify_model, roi_imgs)
                for idx, status in enumerate(preds_list):
                    student_answers[roi_labels[idx]] = status

            stats = evaluate_recognition_performance(student_answers, gt_answer_types, q_configs, roi_labels, roi_indices)
            fold_total_q += stats['total_questions']
            fold_correct_q += stats['correct_questions']

            all_detailed_reports.append({
                'Fold': fold,
                'Image': image_name,
                'Exam': exam_id,
                'Page': page_num,
                'Total_Q': stats['total_questions'],
                'Correct_Q': stats['correct_questions'],
                'Wrong_Log': "; ".join(stats['wrong_details'])
            })

            # Xử lý gộp phiếu theo logic 
            if exam_id == 0:
                try:
                    base = image_name.rsplit('.', 1)[0]
                    sheet_id = base.rsplit('_', 1)[0]
                except: sheet_id = image_name

                if sheet_id not in exam0_sheets:
                    exam0_sheets[sheet_id] = {'pages': 0, 'correct_q': 0, 'total_q': 0}
                exam0_sheets[sheet_id]['pages'] += 1
                exam0_sheets[sheet_id]['correct_q'] += stats['correct_questions']
                exam0_sheets[sheet_id]['total_q'] += stats['total_questions']
            else:
                actual_total_sheets += 1
                if stats['correct_questions'] == stats['total_questions']:
                    actual_perfect_sheets += 1

        # Xử lý tổng kết riêng cho Exam 0
        for sheet_id, data in exam0_sheets.items():
            if data['pages'] == 3:
                actual_total_sheets += 1
                if data['correct_q'] == data['total_q']:
                    actual_perfect_sheets += 1

        q_acc = (fold_correct_q / fold_total_q * 100) if fold_total_q > 0 else 0
        s_acc = (actual_perfect_sheets / actual_total_sheets * 100) if actual_total_sheets > 0 else 0

        print(f"📊 Kết quả Fold {fold}:")
        print(f"   - Question Accuracy: {q_acc:.2f}% ({fold_correct_q}/{fold_total_q})")
        print(f"   - Sheet Accuracy:    {s_acc:.2f}% ({actual_perfect_sheets}/{actual_total_sheets})")

        fold_results.append({'Fold': fold, 'Q_Acc': q_acc, 'S_Acc': s_acc})

    # =============================================================================
    # --- 4. XUẤT 2 FILE BÁO CÁO VÀ TÍNH TRUNG BÌNH ĐỘ LỆCH CHUẨN ---
    # =============================================================================
    if len(all_detailed_reports) > 0:
        df_reports = pd.DataFrame(all_detailed_reports)
        df_reports.to_csv(REPORT_OUTPUT_FILE, index=False)
        print(f"\n✅ Đã lưu log lỗi chi tiết từng trang tại: {REPORT_OUTPUT_FILE}")

    # B. Xuất Báo Cáo Tổng Hợp 5 Fold
    if len(fold_results) > 0:
        df_folds = pd.DataFrame(fold_results)
        q_acc_mean = df_folds['Q_Acc'].mean()
        q_acc_std = df_folds['Q_Acc'].std()
        s_acc_mean = df_folds['S_Acc'].mean()
        s_acc_std = df_folds['S_Acc'].std()

        print("\n🏆 TỔNG KẾT 5-FOLD CROSS VALIDATION 🏆")
        print("="*50)
        print(f"Độ chính xác Câu hỏi (Q_Acc): {q_acc_mean:.2f}% \u00b1 {q_acc_std:.2f}%")
        print(f"Độ chính xác Phiếu   (S_Acc): {s_acc_mean:.2f}% \u00b1 {s_acc_std:.2f}%")
        print("="*50)

        df_folds.to_csv(SUMMARY_OUTPUT_FILE, index=False)

        # Append the summary statistics to the same CSV file
        summary_data = {
            'Fold': ['Mean', 'Std Dev'],
            'Q_Acc': [q_acc_mean, q_acc_std],
            'S_Acc': [s_acc_mean, s_acc_std]
        }
        df_summary = pd.DataFrame(summary_data)
        df_summary.to_csv(SUMMARY_OUTPUT_FILE, mode='a', header=False, index=False)

        print(f"✅ Đã lưu bảng tổng hợp từng fold tại: {SUMMARY_OUTPUT_FILE}")
    else:
        print("❌ Không có dữ liệu. Vui lòng kiểm tra lại đường dẫn BASE_FOLDS_DIR!")

if __name__ == '__main__':
    main()

🚀 BẮT ĐẦU ĐÁNH GIÁ 5-FOLD CROSS VALIDATION TRÊN TOÀN PHIẾU...
Loading Metadata from /content/drive/MyDrive/OMR-Datasets/metadata/modelAnswerMetadata.mat...
Loading Student Exams from /content/drive/MyDrive/OMR-Datasets/metadata/exams.mat...

🔍 ĐANG ĐÁNH GIÁ FOLD 1 / 5
Loading EfficientNet-B0 from /content/drive/MyDrive/OMR-Datasets/train-cls/scene2/EfficientNetB0/Fold_1/efficientnet_b0_fold1_gan.pth...
📊 Kết quả Fold 1:
   - Question Accuracy: 97.99% (2092/2135)
   - Sheet Accuracy:    86.39% (127/147)

🔍 ĐANG ĐÁNH GIÁ FOLD 2 / 5
Loading EfficientNet-B0 from /content/drive/MyDrive/OMR-Datasets/train-cls/scene2/EfficientNetB0/Fold_2/efficientnet_b0_fold2_gan.pth...
📊 Kết quả Fold 2:
   - Question Accuracy: 99.14% (2186/2205)
   - Sheet Accuracy:    90.48% (133/147)

🔍 ĐANG ĐÁNH GIÁ FOLD 3 / 5
Loading EfficientNet-B0 from /content/drive/MyDrive/OMR-Datasets/train-cls/scene2/EfficientNetB0/Fold_3/efficientnet_b0_fold3_gan.pth...
📊 Kết quả Fold 3:
   - Question Accuracy: 98.92% (2188/2212)

In [ ]:
import os

def count_files_in_directory(directory_path):
    """Counts the number of files in a given directory."""
    if not os.path.isdir(directory_path):
        return f"Error: Directory '{directory_path}' not found."

    file_count = 0
    for root, dirs, files in os.walk(directory_path):
        file_count += len(files)
    return file_count

# Example usage: count files in the unzipped dataset directory
directory_to_count = '/content/content/OMR_5Fold_Sheets_TestOnly/Fold_1/test' # Using the BASE_FOLDS_DIR from previous cells
num_files = count_files_in_directory(directory_to_count)
print(f"Number of files in '{directory_to_count}': {num_files}")

Number of files in '/content/content/OMR_5Fold_Sheets_TestOnly/Fold_1/test': 175
